# Climb the Auditable Leaderboard

[auditable](https://github.com/yzhao062/auditable) is a library of auditing methods for AI agents, with leaderboards that reproduce the boards of [CatchBench](https://github.com/yzhao062/catchbench). This notebook reruns the board for failure detection on SWE-Gym trajectories with one call, then places a detector of your own on it, scored on the same runs under the same protocol as every other row.

## Install

`auditable[bench]` installs the methods together with PyOD, scikit-learn, NetworkX, and [agentpile](https://github.com/yzhao062/agentpile), which reads the public SWE-Gym trajectories into one record.

In [ ]:
!pip install -q "auditable[bench]"

## Graph Rows (Optional)

Six of the 17 rows (the PyGOD detectors, GUARDIAN-Lite, and G-Safeguard-Lite) need `auditable[graph-ad]` and a PyG sampling backend built for the installed torch, which pip cannot resolve by itself. Without them the board still runs: those rows are listed as skipped, with the missing extra named.

To add them, uncomment the next cell and run it before the board. Colab already has torch, so the cell keeps it and picks the `pyg-lib` wheel that matches its version.

In [ ]:
# Optional graph rows. Uncomment and run this cell before the board cell.
# import torch
# ver = torch.__version__.split("+")[0]
# tag = "cu" + torch.version.cuda.replace(".", "") if torch.version.cuda else "cpu"
# !pip install -q pyg_lib -f https://data.pyg.org/whl/torch-{ver}+{tag}.html
# !pip install -q "auditable[graph-ad]"
# !python -c "import torch_geometric.typing as t; assert t.WITH_PYG_LIB"

## Run the Board

`leaderboard("detect/swegym")` scores every registered run detector on 376 SWE-Gym OpenHands runs (188 failed) and ranks the rows by ROC-AUC. Supervised methods get the mean over five seeds of stratified 5-fold cross-validation with a fresh estimator per fold, unsupervised methods get one `fit_score` over all runs, and the random row gets five draws from one seed. The first call downloads the trajectories through agentpile, and later calls load them from the cache in a few seconds. The full 17-row board takes a minute or two on a laptop CPU, and the callback below prints each row as it finishes.

In [ ]:
from auditable.bench import leaderboard


def show(row):
    print(f"{row.label:<24} {row.status:<8} {row.seconds:5.1f}s")


board = leaderboard("detect/swegym", progress=show)
print()
print(board)

`board.compare_golden()` compares each score with the CatchBench board value, rounded to three decimals. A value known to differ by platform is reported as such and not as a mismatch: the CatchBench board was computed on Linux, and `run.pyod_lof` prints 0.582 on macOS instead of 0.584.

In [ ]:
checks = board.compare_golden()
known = [c for c in checks if c["platform"]]
bad = [c for c in checks if not c["ok"]]
print(f"{len(checks) - len(bad)} of {len(checks)} scored rows reproduce their CatchBench reference "
      f"({len(known)} at a known platform value).")
for c in bad:
    print("mismatch:", c["key"], c["got"], "vs", c["golden"])

## Read the Board Against the Size Baseline

`run.lr_flat` is logistic regression on run size alone (step, tool-call, decision, and agent counts, the GRADE `flat` layer), so every other row has to be read against it. It is a linear baseline. The board also carries the flexible size control `run.lr_spline_flat`, a cubic spline of the same four counts, which ranks second at 0.832, above both GRADE dependency rows. The top row is a literature baseline (PyGOD GAAN), and the board does not support the reading that the dependency layers predict failure beyond run size, which GRADE retired on 2026-09-08. Read your own row against both size controls.

## Put Your Own Method on the Board

A run detector subclasses `auditable.base.BaseRunDetector`, declares a `MethodCard`, and returns a higher score for a run that more likely failed. Its input is a list of runs, each a typed graph that `normalize_trace` turns into steps with `kind` (`"decision"` or `"tool_call"`), `agent`, and `deps`. The card picks the protocol: `supervised=True` gets `fit(X, y)` on training runs and `decision_function(X)` on held-out runs, as in the board above, and `supervised=False` gets one `fit_score(X)` over all runs.

The detector below is a decision tree on two numbers per run, the share of steps that are tool calls and the log of the step count. It is a template for your own features or model.

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

from auditable.base import BaseRunDetector, MethodCard
from auditable.featurize import normalize_trace


class ToolShareTree(BaseRunDetector):
    """Decision tree on the tool-call share and the log step count of a run."""

    card = MethodCard(
        method_id="run.my_tool_share_tree",
        name="Tool-call share tree",
        family="run",
        fit_scope="inductive",
        supervised=True,
        output_kind="score",
        fidelity="original",
        requires=frozenset({"kind"}),
        extras=("tabular-ad",),
        summary="Decision tree on the share of tool calls and the log step count of a run.",
    )

    def __init__(self, max_depth=2):
        self.max_depth = max_depth

    @staticmethod
    def _features(runs):
        rows = []
        for run in runs:
            kinds = [step["kind"] for step in normalize_trace(run).steps]
            rows.append([np.mean([k == "tool_call" for k in kinds]), np.log(len(kinds))])
        return np.array(rows)

    def fit(self, X, y=None):
        self.tree_ = DecisionTreeClassifier(max_depth=self.max_depth, class_weight="balanced", random_state=0)
        self.tree_.fit(self._features(X), y)
        self.decision_scores_ = self.decision_function(X)
        return self

    def decision_function(self, X):
        return self.tree_.predict_proba(self._features(X))[:, 1]

### Iterate Fast

While you develop, score only your method next to the size baseline. The population is already loaded, so this takes seconds. The board population is also the test set, so choose hyperparameters on a split of your own and enter one final row. A row with constructor arguments or a variant of the same class goes in as `Entry(ToolShareTree, {"max_depth": 3}, label="my tree, depth 3", key="run.my_tool_share_tree.d3")`.

In [ ]:
quick = leaderboard("detect/swegym", methods=["run.lr_flat", ToolShareTree])
print(quick)

### Rank on the Full Board

`extra=` adds your rows to the default set, so the board is scored again with your method ranked among the others. The Markdown table is the board in a form you can paste into an issue, a pull request, or a README.

In [ ]:
board = leaderboard("detect/swegym", extra=[ToolShareTree], progress=show)

key = ToolShareTree.card.method_id
print()
print(f"{key}: rank {board.rank_of(key)} of {len(board.ranked())}, ROC-AUC {board[key].scores['roc_auc']:.3f}")
print(f"run.lr_flat (size baseline): rank {board.rank_of('run.lr_flat')}")
print()
print(board.to_markdown())

### What the Rank Means

The tree reads only counts taken from the run (the share of tool calls and the step count), and it ranks above `run.lr_flat` without reading a single dependency. Clearing the size baseline therefore says little about dependency structure. A claim for your own method needs a comparison with a size-only model fitted as flexibly as yours.

## More Boards

Six boards ship in this release, and each runs through the same call, `leaderboard(task_id)`.

| Task | Question | Metrics |
|---|---|---|
| `detect/swegym` | Did the run fail? SWE-Gym OpenHands trajectories, 376 runs | ROC-AUC |
| `detect/tau` | Did the run fail? tau-bench, 660 runs | ROC-AUC |
| `localize/whoandwhen` | Which step was the decisive mistake? Who&When, 126 failed multi-agent runs | top-1, top-3, MRR |
| `live/swegym` | Can the failure be caught early? Prefixes of 25, 50, 75, and 100 percent of the steps of a run | mean prefix ROC-AUC |
| `live/tau` | Can the failure be caught early? Same prefixes on tau-bench | mean prefix ROC-AUC |
| `pre/over_privilege` | Is the agent configuration over-privileged before it runs? 1,187 configurations from six sources | F1, precision, recall |

The localization and PRE boards include LLM judges, replayed from committed caches with no API call. Each board also carries its own trivial rows (random scores, a position prior, flagging every configuration, run size), and a new row is best read against them.

From a shell, `python -m auditable.bench` lists the tasks, and a task id runs its board:

```
python -m auditable.bench
python -m auditable.bench detect/tau --markdown
python -m auditable.bench localize/whoandwhen --golden
```

`--golden` compares the board with CatchBench and exits with status 1 on a mismatch. The full reference is in the [documentation](https://auditable-ai.readthedocs.io/), and the [README](https://github.com/yzhao062/auditable/blob/main/README.md) covers the methods and the install options.